# Phase 4 Decoder Prompting Baselines



This Colab workflow runs decoder prompting baselines for PERIAD authorship attribution. Full decoder inference should run in Colab; local VS Code is for code checks, smoke validation, aggregation, and plotting.

## Runtime Notes

- Full decoder runs should be run one at a time.
- Llama 3 and Gemma 2 may require gated Hugging Face access.
- Runs are resumable from `predictions.csv`.
- If Colab disconnects, rerun the same command without `--overwrite`.
- Google Drive syncing can take several minutes.
- Large checkpoints should remain outside Git.
- Backup after every major decoder run to avoid Colab interruption loss.

## 1. Clone or Pull the Repository

In [2]:
%%bash
cd /content
if [ ! -d Authorship-Attribution ]; then
  mkdir -p Authorship-Attribution/repo
fi
cd /content/Authorship-Attribution/repo
if [ ! -d Authorship-Attribution-in-Victorian-Periodicals ]; then
  git clone https://github.com/IamTaoHu/Authorship-Attribution-in-Victorian-Periodicals.git Authorship-Attribution-in-Victorian-Periodicals
fi
cd Authorship-Attribution-in-Victorian-Periodicals
git pull

Already up to date.


## Mount Google Drive

In [4]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: User cancelled dfs_ephemeral authorization

In [ ]:
import os
from pathlib import Path

DRIVE_ROOT = "/content/drive/MyDrive/Authorship-Attribution"

BACKUP_DIRS = [
    "artifacts",
    "checkpoints",
    "datasets",
    "exports",
    "repo"
]

for name in BACKUP_DIRS:
    Path(f"{DRIVE_ROOT}/{name}").mkdir(parents=True, exist_ok=True)

print("Google Drive backup folders ready.")

## 2. Check GPU

In [ ]:
!nvidia-smi
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO CUDA")

## 3. Install Requirements

In [ ]:
%cd /content/Authorship-Attribution/repo/Authorship-Attribution-in-Victorian-Periodicals
!pip install -r requirements.txt

## 4. Authenticate with Hugging Face



Store your token in Colab Secrets as `HF_TOKEN` before running gated models.

In [ ]:
from huggingface_hub import login
import os
import getpass

token = getpass.getpass("Enter HF token: ")

os.environ["HF_TOKEN"] = token
login(token=token)

In [ ]:
from huggingface_hub import whoami
whoami()

In [ ]:
from huggingface_hub import model_info

model_info("mistralai/Mistral-7B-Instruct-v0.3")

In [ ]:
model_info("meta-llama/Meta-Llama-3-8B-Instruct")

## 5. Write Colab Path Config



The Colab workspace root is `/content/Authorship-Attribution`.

In [ ]:
%%bash
cd /content/Authorship-Attribution/repo/Authorship-Attribution-in-Victorian-Periodicals
cat > configs/paths.local.yaml <<'YAML'
paths:
  workspace_root: /content/Authorship-Attribution
  repo_root: /content/Authorship-Attribution/repo/Authorship-Attribution-in-Victorian-Periodicals
  artifacts_root: /content/Authorship-Attribution/artifacts
  checkpoints_root: /content/Authorship-Attribution/checkpoints
  datasets_root: /content/Authorship-Attribution/datasets
  exports_root: /content/Authorship-Attribution/exports
runtime:
  default_environment: colab
  cloud_training_environment: colab
YAML

If datasets already exist in Google Drive, restore them before running Phase 4.

In [ ]:
!rsync -avh --ignore-existing \
/content/drive/MyDrive/Authorship-Attribution/datasets/ \
/content/Authorship-Attribution/datasets/

## 6. Prepare or Upload PERIAD Processed Split



Phase 4 expects these files outside Git:



```text

/content/Authorship-Attribution/datasets/processed/periad/train.csv

/content/Authorship-Attribution/datasets/processed/periad/test.csv

/content/Authorship-Attribution/datasets/processed/periad/label_map.json

```



If they are not already present, run the Phase 1 preparation command from the repo documentation before starting Phase 4.

In [ ]:
%%bash
set -e
test -f /content/Authorship-Attribution/datasets/processed/periad/train.csv
test -f /content/Authorship-Attribution/datasets/processed/periad/test.csv
test -f /content/Authorship-Attribution/datasets/processed/periad/label_map.json
ls -lh \
  /content/Authorship-Attribution/datasets/processed/periad/train.csv \
  /content/Authorship-Attribution/datasets/processed/periad/test.csv \
  /content/Authorship-Attribution/datasets/processed/periad/label_map.json

## 7. TinyLlama Smoke

In [ ]:
%cd /content/Authorship-Attribution/repo/Authorship-Attribution-in-Victorian-Periodicals
!python scripts/run_phase4_prompting.py --config configs/phase4/tinyllama_smoke.yaml --overwrite

In [ ]:
!rsync -avh --delete \
/content/Authorship-Attribution/artifacts/phase4/ \
/content/drive/MyDrive/Authorship-Attribution/artifacts/phase4/

## 8. Mistral Zero-Shot

In [ ]:
%cd /content/Authorship-Attribution/repo/Authorship-Attribution-in-Victorian-Periodicals
!python scripts/run_phase4_prompting.py --config configs/phase4/mistral_zero_shot.yaml --overwrite

In [ ]:
!rsync -avh --delete \
/content/Authorship-Attribution/artifacts/phase4/ \
/content/drive/MyDrive/Authorship-Attribution/artifacts/phase4/

## 9. Mistral Few-Shot

In [ ]:
%cd /content/Authorship-Attribution/repo/Authorship-Attribution-in-Victorian-Periodicals
!python scripts/run_phase4_prompting.py --config configs/phase4/mistral_few_shot.yaml

In [ ]:
!rsync -avh --delete \
/content/Authorship-Attribution/artifacts/phase4/ \
/content/drive/MyDrive/Authorship-Attribution/artifacts/phase4/

## 10. Llama 3 Zero-Shot

In [ ]:
%cd /content/Authorship-Attribution/repo/Authorship-Attribution-in-Victorian-Periodicals
!python scripts/run_phase4_prompting.py --config configs/phase4/llama3_zero_shot.yaml

In [ ]:
!rsync -avh --delete \
/content/Authorship-Attribution/artifacts/phase4/ \
/content/drive/MyDrive/Authorship-Attribution/artifacts/phase4/

## 11. Llama 3 Few-Shot

In [ ]:
%cd /content/Authorship-Attribution/repo/Authorship-Attribution-in-Victorian-Periodicals
!python scripts/run_phase4_prompting.py --config configs/phase4/llama3_few_shot.yaml

In [ ]:
!rsync -avh --delete \
/content/Authorship-Attribution/artifacts/phase4/ \
/content/drive/MyDrive/Authorship-Attribution/artifacts/phase4/

## 12. Gemma 2 Zero-Shot

In [ ]:
%cd /content/Authorship-Attribution/repo/Authorship-Attribution-in-Victorian-Periodicals
!python scripts/run_phase4_prompting.py --config configs/phase4/gemma2_zero_shot.yaml

In [ ]:
!rsync -avh --delete \
/content/Authorship-Attribution/artifacts/phase4/ \
/content/drive/MyDrive/Authorship-Attribution/artifacts/phase4/

## 13. Gemma 2 Few-Shot

In [ ]:
%cd /content/Authorship-Attribution/repo/Authorship-Attribution-in-Victorian-Periodicals
!python scripts/run_phase4_prompting.py --config configs/phase4/gemma2_few_shot.yaml

In [ ]:
!rsync -avh --delete \
/content/Authorship-Attribution/artifacts/phase4/ \
/content/drive/MyDrive/Authorship-Attribution/artifacts/phase4/

## 14. Aggregate Results

In [ ]:
%cd /content/Authorship-Attribution/repo/Authorship-Attribution-in-Victorian-Periodicals
!python -m src.evaluation.aggregate_phase4_results

## 15. Plot Results

In [ ]:
%cd /content/Authorship-Attribution/repo/Authorship-Attribution-in-Victorian-Periodicals
!python src/visualization/plot_phase4_results.py

## 16. Check Outputs

In [ ]:
%cd /content/Authorship-Attribution/repo/Authorship-Attribution-in-Victorian-Periodicals
!python scripts/check_phase4_outputs.py --phase4_dir /content/Authorship-Attribution/artifacts/phase4

## Final Google Drive Backup

In [ ]:
!rsync -avh --delete \
/content/Authorship-Attribution/artifacts/ \
/content/drive/MyDrive/Authorship-Attribution/artifacts/

## 17. Zip Artifacts

In [ ]:
%%bash
cd /content/Authorship-Attribution
zip -r phase4_artifacts.zip artifacts/phase4

## 18. Download Artifacts

In [ ]:
from google.colab import files
files.download('/content/Authorship-Attribution/phase4_artifacts.zip')